In [13]:
import pandas as pd
import numpy as np
import psycopg2
import sqlalchemy as db
from sqlalchemy import create_engine
import yaml

In [14]:
with open('C:\\Users\\dylan\\OneDrive\\Documentos\\mensajeria\\config.yml', 'r') as f:
    config = yaml.safe_load(f)
    config_co = config['mensajeria']
    config_etl = config['ETL_PRO']

# Construct the database URL
url_co = (f"{config_co['drivername']}://{config_co['user']}:{config_co['password']}@{config_co['host']}:"
          f"{config_co['port']}/{config_co['dbname']}")
url_etl = (f"{config_etl['drivername']}://{config_etl['user']}:{config_etl['password']}@{config_etl['host']}:"
           f"{config_etl['port']}/{config_etl['dbname']}")
# Create the SQLAlchemy Engine
mensajeria = create_engine(url_co)
etl_conn = create_engine(url_etl)

In [15]:
tabla_cliente = pd.read_sql_table('cliente', mensajeria)
tabla_cliente.groupby('cliente_id')
tabla_cliente.head(10)

,cliente_id,nit_cliente,nombre,email,direccion,telefono,nombre_contacto,ciudad_id,tipo_cliente_id,activo,coordinador_id,sector
0,1,25,Cliente 2,algo.com,Calle 100 No 25-18,327-00000,Cristiano Ronaldo,1,1,True,NaN,S
1,2,123,Cliente 1,algo.com,Calle 100 No 25-18,327-00000,Cristiano Ronaldo,1,1,True,2.0,industrial
2,6,24390-3,CLINICA DEPORTIVA DEL SUR,algo.com,Calle 100 No 25-18,327-00000,Cristiano Ronaldo,1,1,True,1.0,salud
3,19,8301821,HOSPITAL ORTOPEDICO DE COLOMBIA,algo.com,Calle 100 No 25-18,327-00000,Cristiano Ronaldo,1,1,True,NaN,salud
4,8,5017350-8,CLINICA NEFROLOGOS DE CALI,algo.com,Calle 100 No 25-18,327-00000,Cristiano Ronaldo,1,1,True,NaN,salud
5,3,312289-5,BANCO REGIONAL DE SANGRE BLOD-LIFE,algo.com,Calle 100 No 25-18,327-00000,Cristiano Ronaldo,1,1,True,NaN,salud
6,4,306215-0,CRUZ AZUL-LIFE,algo.com,Calle 100 No 25-18,327-00000,Cristiano Ronaldo,1,1,True,NaN,salud
7,5,300513-3,CLINICA CALI -JOVEN,algo.com,Calle 100 No 25-18,327-00000,Cristiano Ronaldo,1,1,True,NaN,salud
8,7,951033-8,CLINICA COFFE -HEALTH,algo.com,Calle 100 No 25-18,327-00000,Cristiano Ronaldo,1,1,True,NaN,salud
9,9,149757-6,UNIDAD DE TRAUMA DEL OESTE,algo.com,Calle 100 No 25-18,327-00000,Cristiano Ronaldo,1,1,True,NaN,salud


In [16]:
tabla_sede = pd.read_sql_table('sede', mensajeria)
tabla_sede.head(10)

,sede_id,nombre,direccion,telefono,nombre_contacto,ciudad_id,cliente_id
0,10,FARALLONES /123,Los angeles distrito Latino,310-70000,JUAN PEREZ,1,4
1,11,REMEDIOZ/ 123,Los angeles distrito Latino,310-70000,JUAN PEREZ,1,4
2,13,DIME / LOS ROJOS,Los angeles distrito Latino,310-70000,JUAN PEREZ,1,4
3,14,DESPACHOS / LOS ROJOS,Los angeles distrito Latino,310-70000,JUAN PEREZ,1,4
4,23,POPAYAN BODEGA 28 / A,Los angeles distrito Latino,310-70000,JUAN PEREZ,3,11
5,24,PASTO VITRINA /,Los angeles distrito Latino,310-70000,JUAN PEREZ,4,11
6,25,PASTO BODEGA 29/,Los angeles distrito Latino,310-70000,JUAN PEREZ,4,11
7,26,PALMIRA BODEGA 20 /,Los angeles distrito Latino,310-70000,JUAN PEREZ,2,11
8,32,CLINICA FABILUX,Los angeles distrito Latino,310-70000,JUAN PEREZ,1,12
9,33,FORDES PASOANCHOS,Los angeles distrito Latino,310-70000,JUAN PEREZ,1,11


In [17]:
dim_proveedor = pd.merge(tabla_cliente,tabla_sede,left_on='cliente_id',right_on='cliente_id',how='left')
dim_proveedor.rename(columns={'cliente_id':'id_proveedor','nombre':'nombre_proveedor'}, inplace=True)
#Se eliminan datos de la tabla clientes
dim_proveedor.drop(columns=['email','telefono_x','direccion_x','nombre_contacto_x','ciudad_id_x','tipo_cliente_id','activo','coordinador_id'], inplace=True)
#se eliminan los datos de la tabla sede
dim_proveedor.drop(columns=['telefono_y','direccion_y','nombre_contacto_y','ciudad_id_y','sede_id'], inplace=True)
dim_proveedor.rename(columns={'nombre_y':'nombre_sede','nombre_x':'nombre_proveedor'},inplace=True)
dim_proveedor['key_proveedor']=range(1,len(dim_proveedor)+1)
dim_proveedor.head(10)


,id_proveedor,nit_cliente,nombre_proveedor,sector,nombre_sede,key_proveedor
0,1,25,Cliente 2,S,NaN,1
1,2,123,Cliente 1,industrial,Sede principal - Cliente1,2
2,2,123,Cliente 1,industrial,sede aux - cliente 1,3
3,6,24390-3,CLINICA DEPORTIVA DEL SUR,salud,PRINCIPAL /14,4
4,19,8301821,HOSPITAL ORTOPEDICO DE COLOMBIA,salud,NaN,5
5,8,5017350-8,CLINICA NEFROLOGOS DE CALI,salud,HEMAS,6
6,8,5017350-8,CLINICA NEFROLOGOS DE CALI,salud,NUEVA HEMATO,7
7,8,5017350-8,CLINICA NEFROLOGOS DE CALI,salud,NUEVA HEMATO,8
8,8,5017350-8,CLINICA NEFROLOGOS DE CALI,salud,NUEVA HEMATO,9
9,3,312289-5,BANCO REGIONAL DE SANGRE BLOD-LIFE,salud,HEMOLIFE,10


## AQUI SE HARIA LIMPIEZA DE DATOS